In [55]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


setting_up_proxy_from_config()

setting up proxy 'http://127.0.0.1:7890' for 'http'
setting up proxy 'http://127.0.0.1:7890' for 'https'
setting up proxy 'socks5://127.0.0.1:7890' for 'all'



In [59]:
import requests

resp = requests.get('https://civitai.com/models/6433/loraflatcolor', proxies=default_proxy_config)
print(resp)

<Response [200]>


In [8]:
with open('examples/naive_model_page.html', 'w') as f:
    f.write(resp.text)

In [30]:
import json
import importlib
import RFC.utils.parse
from RFC.utils.parse import (
    parse,
    get_html_soup
)
importlib.reload(RFC.utils.parse)

soup = get_html_soup(resp.text)
# parse_config = {
#     ('type', 'script', -3): {
#     }
# }
parse_config = {
    ('attr', 'script', 'id', '__NEXT_DATA__', -1): {
        ('result', 'text', None): {}
    }
}
result = parse(soup, parse_config, return_str=True, debug=True)
json.dump(json.loads(result[0]), open('examples/model.json', 'w'), indent=4, ensure_ascii=False)

<!DOCTYPEhtml><htmllang="en"><head><metacharset="utf-8"/><metacontent="width=device-width"name="view...
└── <scriptid="__NEXT_DATA__"type="application/json">{"props":{"pageProps":{"colorScheme":"dark","cookie...
    └── {"props":{"pageProps":{"colorScheme":"dark","cookies":{"models":{"types":[],"baseModels":[],"status"...


In [ ]:
def _cell_analysis_1():
    """
        we can get gallery ids at this step, from some specific author
    """
    import requests
    resp = requests.get('https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi', proxies=default_proxy_config)
    print(resp)
    print(resp.content)
    print(resp.headers['content-type'])

    def decode_gallery_id(resp):
        bytes = resp.content
        assert len(bytes) % 4 == 0, f'length of bytes {len(bytes)} not divisible by 4'
        ids = []
        for i in range(0, len(bytes), 4):
            ids.append(int.from_bytes(bytes[i:i+4], 'big'))
        return ids

    print(decode_gallery_id(resp))

_cell_analysis_1()

In [ ]:
def _cell_analysis_2():
    """
        we can get gallery links, names and metadata(tags, language, type, series, author) at this step
    """
    import urllib.parse as urlparse
    import requests
    resp = requests.get('https://ltn.hitomi.la/galleryblock/2813534.html', proxies=default_proxy_config, headers={'referer': 'https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi'})
    print(resp)
    print(resp.content)
    with open('ex.html', 'wb') as f:
        f.write(resp.content)
    print()

    from RFC.utils.parse import (
        BeautifulSoup,
        parse,
    )
    
    def parse_gallery_block(resp):
        parse_config = {
            'all_links': {
                ('result', 'href', (None,)): {},
            },
        }
        return parse(BeautifulSoup(resp.content), parse_config['all_links'])

    def get_gallery_link(links):
        return urlparse.urljoin('https://hitomi.la/', links[0].attrs['href'])

    def get_metadata(links):
        metadata = {}
        for link in links:
            if link.attrs['href'].startswith('/tag/'):
                if 'tag' not in metadata:
                    metadata['tag'] = []
                metadata['tag'].append(link.text)
            if link.attrs['href'].startswith('/series/'):
                if 'series' not in metadata:
                    metadata['series'] = []
                metadata['series'].append(link.text)
            if link.attrs['href'].startswith('/type/'):
                if 'type' not in metadata:
                    metadata['type'] = []
                metadata['type'].append(link.text)
            if link.attrs['href'].startswith('/artist/'):
                if 'artist' not in metadata:
                    metadata['artist'] = []
                metadata['artist'].append(link.text)
            if link.attrs['href'].startswith('/index-'):
                if 'language' not in metadata:
                    metadata['language'] = []
                metadata['language'].append(link.text)
        return metadata


    links = parse_gallery_block(resp)
    gallery_link = get_gallery_link(links)
    metadata = get_metadata(links)
    
    import json
    print(gallery_link, json.dumps(metadata, indent=4, ensure_ascii=False), sep='\n\n')

_cell_analysis_2()

In [ ]:
import re
import time
import requests


while True:
    try:
        resp = requests.get('https://ltn.hitomi.la/gg.js', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/509748.html'})
        result = re.findall('b: \'([0-9]*)\/', resp.text)
        assert len(result) == 1
        break
    except Exception as e:
        time.sleep(1)
        
resp.text

In [ ]:
import re
import json
import time
import pickle
import requests

def _request_prefix():
    while True:
        try:
            resp = requests.get('https://ltn.hitomi.la/gg.js', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/509748.html'})
            prefix = re.findall('b: \'([0-9]*)\/', resp.text)
            value_o = [int(x) for x in re.findall('o = ([0-1])', resp.text)]
            hit_hash = [int(x) for x in re.findall('case ([0-9]+):', resp.text)]
            assert len(prefix) == 1 and len(value_o) == 2
            return prefix[0], value_o, hit_hash
        except Exception as e:
            time.sleep(1)
    
def _from_hash_to_url(hash, prefix):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'
    # '1706601602/2151/b5ddc3da968890e7ef4b94982e286a8cd27319c9fbc4853012d5be6546ebc678'
    
def _get_base(value_o, hit_hash, hash):
    o = value_o[-1] if int(hash[-1]+hash[-3:-1], 16) in hit_hash else value_o[0]
    return chr(ord('a')+o)

def _add_image(id, img_type, img_hash, prefix, base):
    url = f'https://{base}a.hitomi.la/{img_type}/{_from_hash_to_url(img_hash, prefix)}.{img_type}'
    # https://aa.hitomi.la/avif/1706626802/824/55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383.avif

result = pickle.load(open('saves/mutou mato/subs/1004701/_result.pkl', 'rb'))
# print(result)
js = result['response'].text
assert js.startswith('var galleryinfo = ')
js = json.loads(js[len('var galleryinfo = '):])
prefix, value_o, hit_hash = _request_prefix()
for id, img in enumerate(js['files']):
    hash = img['hash']
    id = f'{id:04d}'
    added = False
    base = _get_base(value_o, hit_hash, hash)
    if 'hasavif' in img and img['hasavif'] == 1:
        _add_image(id, 'avif', hash, prefix, base)
        added = True
    if 'haswebp' in img and img['haswebp'] == 1:
        _add_image(id, 'webp', hash, prefix, base)
        added = True
    if not added and 'hasjxl' in img and img['hasjxl'] == 1:  # optional
        _add_image(id, 'jxl', hash, prefix, base)
        added = True

In [ ]:
def _cell_analysis_2():

    import urllib.parse as urlparse
    import requests
    resp = requests.get('https://ltn.hitomi.la/galleries/2813534.js', proxies=default_proxy_config, headers={'referer': 'https://ltn.hitomi.la/artist/mutou%20mato-all.nozomi'})
    print(resp)
    print(resp.text)
    
    
_cell_analysis_2()

In [ ]:
def from_hash_to_url(hash, prefix=1706601602):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'

import requests
resp = requests.get('https://ba.hitomi.la/webp/1706637601/1224/d056bc87f4ff4b496048f662a173e4fcc109bbd24782913064810f44293e7c84.webp', proxies=default_proxy_config, headers={'referer': 'https://hitomi.la/reader/2813534.html'})
print(resp)
with open('img.webp', 'wb') as f:
    f.write(resp.content)

In [ ]:
# hash = '55f3b7adf6924bd63d88d1d1d990d2ab3114d03be5adcf682ac56900a41b2383'
# hash = '0a656735699096ac193dab014d78fc2b98f9569a8b0e38505a7880b533a97ca6'
# hash = 'b5ddc3da968890e7ef4b94982e286a8cd27319c9fbc4853012d5be6546ebc678'
def from_hash_to_url(hash, prefix=1706601602):
    return f'{prefix}/{int(hash[-1]+hash[-3:-1], 16)}/{hash}'


from_hash_to_url(hash)